In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType, TimestampType
import pyspark.sql.functions as f
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommerceeastus001", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df = spark.readStream \
    .format('delta') \
    .option('readChangeFeed', 'true') \
    .table(f"{catalog_name}.silver.slv_order_shipments")


In [0]:

df = df.withColumn(
    "carrier_group",
    f.when(f.col("carrier").isin("ECOMEXPRESS", "DELHIVERY", "XPRESSBEES", "BLUEDART"), "Domestic")
     .otherwise("International")
)

In [0]:
df_gold  = df.withColumn(
    "is_weekend_shipment",
    f.when(f.dayofweek("order_dt").isin(1, 7), True)
     .otherwise(False)
)

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_shipments/"
print(gold_checkpoint_path)

In [0]:
def upsert_to_gold(microBatchDF, batch_id):
    table_name = f"{catalog_name}.gold.gld_order_shipments"

    # Drop CDC metadata columns from the Change Data Feed read
    microBatchDF = microBatchDF.drop("_change_type", "_commit_version", "_commit_timestamp")

    if not spark.catalog.tableExists(table_name):
        # 🔹 First time: create table
        (microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name))

        # Enable Change Data Feed (CDF)
        spark.sql(
            f"""
            ALTER TABLE {table_name} SET TBLPROPERTIES (
                delta.enableChangeDataFeed = true
            )
        """
        )

    else:
        # Load existing Delta table
        delta_table = DeltaTable.forName(spark, table_name)

        # Build explicit column mapping to avoid schema mismatch with CDC columns
        column_map = {col: f"batch_table.{col}" for col in microBatchDF.columns}

        # Perform UPSERT (MERGE)
        (
            delta_table.alias("gold_table")
            .merge(
                microBatchDF.alias("batch_table"),
                """
    gold_table.order_id = batch_table.order_id AND
    gold_table.shipment_id = batch_table.shipment_id
    """,
            )
            .whenMatchedUpdate(set=column_map)
            .whenNotMatchedInsert(values=column_map)
            .execute()
        )

In [0]:
df_gold.writeStream.trigger(availableNow=True) \
    .foreachBatch(upsert_to_gold) \
    .option("checkpointLocation", gold_checkpoint_path) \
    .option('mergeSchema', 'true') \
    .outputMode('update') \
    .start() \
    .awaitTermination()